# 🪤 Tripwire: run it in Google Colab

**A voice memory for decisions *and what would change them*.** Built for the Amazon Developer Hackathon 2026
(Alexa+ track + AWS Builder mini challenge).

This notebook is **self-contained**: it writes the whole server to disk, starts the real MCP server (Streamable HTTP) inside
Colab, and drives it with a real MCP client, exactly like Alexa+ would.

**How to use:** *Runtime → Run all.* It works **with no AWS account** (an offline parser stands in for Bedrock).
Sections 7 and 8 are optional: real **Amazon Bedrock + DynamoDB**, and a public URL for **Alexa+**.

| Section | What it does |
|---|---|
| 1-3 | Install, write the source files, configure |
| 4 | Start the MCP server in the background |
| 5 | **Play the full demo conversation** |
| 5b | *(optional)* The simulated Alexa+ screen |
| 6 | **Safety proof:** a lying model cannot flip a decision |
| 7 | *(optional)* Real Bedrock + DynamoDB |
| 8 | *(optional)* Public HTTPS URL to connect Alexa+ |

## 1. Install

In [ ]:
import sys
assert sys.version_info >= (3, 11), f"Tripwire needs Python 3.11+, this runtime has {sys.version.split()[0]}. Use a newer Colab runtime."
!pip -q install "mcp>=2.3,<3" boto3 uvicorn tzdata
!mkdir -p tripwire_colab/src/tripwire

## 2. Write the source files
_These cells are generated from the real repo files, so what you run here is what you submit._

In [ ]:
%%writefile tripwire_colab/src/tripwire/__init__.py
"""Tripwire: a voice memory for decisions and the conditions that would change them."""

__version__ = "0.1.0"

In [ ]:
%%writefile tripwire_colab/src/tripwire/store.py
"""Persistence layer.

Two interchangeable backends behind one tiny document-store interface:

* ``SqliteStore``   - zero-setup local backend (default; used for tests and local demos).
* ``DynamoStore``   - Amazon DynamoDB backend (set ``TRIPWIRE_STORE=dynamodb``).

Every decision (with its tripwires and history) is one small JSON document keyed by
(household, kind, key), so a decision made in one Alexa conversation can be checked
weeks later in another.

Concurrency: the MCP server is stateless and may run as several containers, so two requests can
touch the same decision at once. Every document carries a version number and every write can be
conditional on it (compare-and-swap). ``update()`` wraps the read-modify-write loop and retries on
conflict, so concurrent edits are merged instead of one silently overwriting the other.
"""

from __future__ import annotations

import copy
import json
import os
import sqlite3
import threading
from abc import ABC, abstractmethod
from typing import Any, Callable


class ConflictError(Exception):
    """A conditional write lost a race (someone else changed the document first)."""


class Store(ABC):
    @abstractmethod
    def get_v(self, household: str, kind: str, key: str) -> tuple[dict[str, Any] | None, int]:
        """Return (document or None, version). A missing document has version 0."""

    @abstractmethod
    def put(self, household: str, kind: str, key: str, data: dict[str, Any], expected_ver: int | None = None) -> int:
        """Write a document; if ``expected_ver`` is given the write only succeeds when the stored
        version matches (0 means 'must not exist'). Raises ConflictError otherwise. Returns the new version."""

    @abstractmethod
    def delete(self, household: str, kind: str, key: str) -> None: ...

    @abstractmethod
    def list(self, household: str, kind: str) -> list[dict[str, Any]]: ...

    @abstractmethod
    def purge(self, household: str) -> int:
        """Delete everything stored for a household (demos, tests, data-deletion requests)."""

    @abstractmethod
    def ping(self) -> None:
        """Cheap round-trip to the backing store; raises if it is unreachable."""

    def get(self, household: str, kind: str, key: str) -> dict[str, Any] | None:
        return self.get_v(household, kind, key)[0]

    def update(self, household: str, kind: str, key: str,
               fn: Callable[[dict[str, Any] | None], dict[str, Any] | None], retries: int = 8) -> dict[str, Any] | None:
        """Atomic read-modify-write. ``fn`` receives a private copy of the current document (or None) and
        returns the new document, or None for 'no change'. It may run more than once, so it must be a pure
        function of its argument (collect side results in a local that it overwrites on each call)."""
        for _ in range(retries):
            doc, ver = self.get_v(household, kind, key)
            new = fn(copy.deepcopy(doc) if doc is not None else None)
            if new is None:
                return None
            try:
                self.put(household, kind, key, new, expected_ver=ver)
                return new
            except ConflictError:
                continue
        raise ConflictError(f"could not update {kind}#{key} after {retries} attempts")


class SqliteStore(Store):
    def __init__(self, path: str = "tripwire.db") -> None:
        self._lock = threading.RLock()
        self._db = sqlite3.connect(path, check_same_thread=False, timeout=5)
        self._db.execute(
            "CREATE TABLE IF NOT EXISTS docs ("
            "household TEXT NOT NULL, kind TEXT NOT NULL, key TEXT NOT NULL, "
            "data TEXT NOT NULL, ver INTEGER NOT NULL DEFAULT 1, PRIMARY KEY (household, kind, key))"
        )
        self._db.commit()

    def get_v(self, household, kind, key):
        with self._lock:
            row = self._db.execute(
                "SELECT data, ver FROM docs WHERE household=? AND kind=? AND key=?", (household, kind, key)
            ).fetchone()
        return (json.loads(row[0]), row[1]) if row else (None, 0)

    def put(self, household, kind, key, data, expected_ver=None):
        with self._lock:  # one process-wide lock makes check-then-write atomic for this backend
            row = self._db.execute(
                "SELECT ver FROM docs WHERE household=? AND kind=? AND key=?", (household, kind, key)
            ).fetchone()
            cur = row[0] if row else 0
            if expected_ver is not None and cur != expected_ver:
                raise ConflictError(f"{kind}#{key}: expected v{expected_ver}, found v{cur}")
            self._db.execute(
                "INSERT INTO docs(household, kind, key, data, ver) VALUES (?,?,?,?,?) "
                "ON CONFLICT(household, kind, key) DO UPDATE SET data=excluded.data, ver=excluded.ver",
                (household, kind, key, json.dumps(data), cur + 1),
            )
            self._db.commit()
            return cur + 1

    def delete(self, household, kind, key):
        with self._lock:
            self._db.execute("DELETE FROM docs WHERE household=? AND kind=? AND key=?", (household, kind, key))
            self._db.commit()

    def list(self, household, kind):
        with self._lock:
            rows = self._db.execute(
                "SELECT key, data FROM docs WHERE household=? AND kind=? ORDER BY key", (household, kind)
            ).fetchall()
        return [{**json.loads(d), "_key": k} for k, d in rows]

    def purge(self, household):
        with self._lock:
            n = self._db.execute("DELETE FROM docs WHERE household=?", (household,)).rowcount
            self._db.commit()
        return n

    def ping(self):
        with self._lock:
            self._db.execute("SELECT 1").fetchone()


class DynamoStore(Store):
    """Single-table DynamoDB design: pk = HOUSEHOLD#<id>, sk = <kind>#<key>; attributes ``data`` (JSON) and ``ver``."""

    def __init__(self, table_name: str, region: str | None = None, client: Any = None) -> None:
        if client is None:
            import boto3  # imported lazily so local mode needs no AWS config
            from botocore.config import Config

            # Bounded latency: a voice request must not hang on a sick network. Retries cover throttling/5xx.
            cfg = Config(
                connect_timeout=float(os.environ.get("TRIPWIRE_DDB_CONNECT_TIMEOUT", "2")),
                read_timeout=float(os.environ.get("TRIPWIRE_DDB_READ_TIMEOUT", "4")),
                retries={"total_max_attempts": int(os.environ.get("TRIPWIRE_DDB_ATTEMPTS", "3")), "mode": "standard"},
            )
            client = boto3.client("dynamodb", region_name=region, config=cfg)
        self._ddb = client
        self._table = table_name

    @staticmethod
    def _pk(household: str) -> dict[str, str]:
        return {"S": f"HOUSEHOLD#{household}"}

    def _key(self, household, kind, key):
        return {"pk": self._pk(household), "sk": {"S": f"{kind}#{key}"}}

    def get_v(self, household, kind, key):
        item = self._ddb.get_item(TableName=self._table, Key=self._key(household, kind, key), ConsistentRead=True).get("Item")
        if not item:
            return None, 0
        return json.loads(item["data"]["S"]), int(item.get("ver", {"N": "1"})["N"])

    def put(self, household, kind, key, data, expected_ver=None):
        kwargs: dict[str, Any] = dict(
            TableName=self._table, Key=self._key(household, kind, key),
            UpdateExpression="SET #d = :d ADD #v :one",
            ExpressionAttributeNames={"#d": "data", "#v": "ver"},
            ExpressionAttributeValues={":d": {"S": json.dumps(data)}, ":one": {"N": "1"}},
            ReturnValues="UPDATED_NEW",
        )
        if expected_ver is not None:
            if expected_ver == 0:
                kwargs["ConditionExpression"] = "attribute_not_exists(pk)"
            else:
                kwargs["ConditionExpression"] = "#v = :e"
                kwargs["ExpressionAttributeValues"][":e"] = {"N": str(expected_ver)}
        try:
            resp = self._ddb.update_item(**kwargs)
        except self._ddb.exceptions.ConditionalCheckFailedException as e:
            raise ConflictError(f"{kind}#{key}: version mismatch") from e
        return int(resp["Attributes"]["ver"]["N"])

    def delete(self, household, kind, key):
        self._ddb.delete_item(TableName=self._table, Key=self._key(household, kind, key))

    def _query(self, household: str, prefix: str | None):
        kwargs: dict[str, Any] = {"TableName": self._table, "ConsistentRead": True,
                                  "KeyConditionExpression": "pk = :pk", "ExpressionAttributeValues": {":pk": self._pk(household)}}
        if prefix:
            kwargs["KeyConditionExpression"] += " AND begins_with(sk, :p)"
            kwargs["ExpressionAttributeValues"][":p"] = {"S": prefix}
        while True:
            resp = self._ddb.query(**kwargs)
            yield from resp.get("Items", [])
            if "LastEvaluatedKey" not in resp:
                return
            kwargs["ExclusiveStartKey"] = resp["LastEvaluatedKey"]

    def list(self, household, kind):
        out = [{**json.loads(i["data"]["S"]), "_key": i["sk"]["S"].split("#", 1)[1]} for i in self._query(household, f"{kind}#")]
        return sorted(out, key=lambda d: d["_key"])

    def purge(self, household):
        n = 0
        for i in list(self._query(household, None)):
            self._ddb.delete_item(TableName=self._table, Key={"pk": i["pk"], "sk": i["sk"]})
            n += 1
        return n

    def ping(self):
        self._ddb.get_item(TableName=self._table, Key=self._key("_health", "ping", "x"))


def make_store() -> Store:
    backend = os.environ.get("TRIPWIRE_STORE", "sqlite").lower()
    if backend == "dynamodb":
        return DynamoStore(os.environ.get("TRIPWIRE_TABLE", "Tripwire"), region=os.environ.get("AWS_REGION"))
    return SqliteStore(os.environ.get("TRIPWIRE_DB", "tripwire.db"))

In [ ]:
%%writefile tripwire_colab/src/tripwire/ai.py
"""Language understanding for Tripwire.

Two jobs, both done through Amazon Bedrock's Converse API with *forced tool use* (the model must
answer by calling a function whose arguments match a JSON schema):

1. ``extract_decision``: turn a spoken, messy decision ("we're going with the 200 meg plan because
   it's cheapest, we'd switch if the price goes over fifty or outages hit more than twice a month")
   into a structured record with explicit, machine-checkable tripwires.
2. ``assess_fact``: given a new fact ("the provider raised the price to $55") and every armed
   tripwire, say which ones it relates to and what number was observed.

The model only *proposes*. ``service.py`` re-validates everything and performs the numeric
comparison itself, so a hallucinated verdict can never silently flip a decision.

If Bedrock is unavailable, a small rule-based fallback keeps the server usable (and the tests
hermetic). Results always say which engine produced them.
"""

from __future__ import annotations

import math
import os
import re
import threading
import time
from typing import Any

OPS = (">", ">=", "<", "<=")
STOP = {"the", "a", "an", "our", "my", "we", "i", "it", "is", "are", "was", "to", "of", "and", "or", "in",
        "on", "for", "that", "this", "than", "per", "now", "has", "have", "had", "with", "at", "be", "will"}

NUMWORDS = {"once": 1, "twice": 2, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
            "seven": 7, "eight": 8, "nine": 9, "ten": 10, "twelve": 12, "twenty": 20, "thirty": 30, "forty": 40,
            "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90, "hundred": 100, "a": 1, "an": 1}
MAX_PROMPT_ITEMS = 60          # cap tripwires sent to the model (cost, latency, context limits)
MAX_TEXT = 1500                # hard cap on any user text we process


def norm(word: str) -> str:
    w = re.sub(r"[^a-z]", "", word.lower())
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"
    if w.endswith("s") and not w.endswith("ss") and len(w) > 3:
        return w[:-1]
    return w


def tokens(text: str) -> set[str]:
    return {norm(w) for w in re.findall(r"[A-Za-z']+", text) if norm(w) and norm(w) not in STOP}


# ------------------------------------------------------------------------------ schemas

EXTRACT_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "Short topic name, 2-5 words, e.g. 'internet plan'."},
        "choice": {"type": "string", "description": "What was decided, e.g. 'the 200 Mbps plan'."},
        "options": {"type": "array", "items": {"type": "string"}, "description": "Alternatives that were considered."},
        "reason": {"type": "string", "description": "Why, in one short sentence."},
        "tripwires": {
            "type": "array",
            "description": "Conditions that would make the person reconsider. Prefer numeric ones.",
            "items": {
                "type": "object",
                "properties": {
                    "description": {"type": "string", "description": "Plain-English, speakable, e.g. 'price goes above $50'."},
                    "metric": {"type": "string", "description": "What is measured, e.g. 'price' or 'outages per month'. Omit if not measurable."},
                    "op": {"type": "string", "enum": list(OPS)},
                    "value": {"type": "number"},
                    "unit": {"type": "string"},
                },
                "required": ["description"],
            },
        },
        "revisit_in_days": {"type": "integer", "description": "Only if the person asked to be reminded or to check back after a period."},
    },
    "required": ["title", "choice", "reason", "tripwires"],
}

ASSESS_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "observations": {
            "type": "array",
            "description": "One entry per tripwire the fact is relevant to. Omit unrelated tripwires.",
            "items": {
                "type": "object",
                "properties": {
                    "tripwire_id": {"type": "string"},
                    "observed_value": {"type": "number", "description": "The number the fact states for this tripwire's metric, if any."},
                    "verdict": {"type": "string", "enum": ["tripped", "not_tripped", "unsure"]},
                    "note": {"type": "string"},
                },
                "required": ["tripwire_id", "verdict"],
            },
        }
    },
    "required": ["observations"],
}

EXTRACT_SYSTEM = (
    "You are Tripwire, a voice assistant that remembers household decisions together with the conditions "
    "that would change them. The user's words come from speech recognition, so expect run-on sentences and "
    "numbers spelled out. Extract the decision and its tripwires by calling submit_decision exactly once. "
    "Convert spoken numbers to digits. Only include tripwires the person actually stated; never invent any. "
    "If a tripwire is a measurable threshold, fill metric/op/value/unit so it can be checked automatically."
)
ASSESS_SYSTEM = (
    "You check whether a newly reported fact trips any of a household's armed tripwires. Call submit_assessment "
    "exactly once. Only mention tripwires the fact is genuinely about (same subject). Report observed_value "
    "whenever the fact states the relevant quantity. Use verdict 'tripped' only when the fact clearly satisfies "
    "the condition, 'not_tripped' when it clearly does not, and 'unsure' when it is ambiguous."
)


# ------------------------------------------------------------------------------ Bedrock


def _bedrock_client():
    import boto3
    from botocore.config import Config

    # Bounded latency: a voice turn must not hang. Worst case ~ total_attempts x read_timeout (default 2 x 6s).
    # NB: botocore's `max_attempts` counts *retries* (so 2 would mean 3 tries); `total_max_attempts` is what we mean.
    cfg = Config(
        connect_timeout=float(os.environ.get("TRIPWIRE_BEDROCK_CONNECT_TIMEOUT", "3")),
        read_timeout=float(os.environ.get("TRIPWIRE_BEDROCK_READ_TIMEOUT", "6")),
        retries={"total_max_attempts": int(os.environ.get("TRIPWIRE_BEDROCK_ATTEMPTS", "2")), "mode": "standard"},
    )
    return boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION"), config=cfg)


def _request(system: str, user: str, tool_name: str, schema: dict[str, Any], model_id: str, with_temperature: bool) -> dict[str, Any]:
    inference: dict[str, Any] = {"maxTokens": 1500}
    if with_temperature:
        inference["temperature"] = 0
    return dict(
        modelId=model_id,
        system=[{"text": system}],
        messages=[{"role": "user", "content": [{"text": user}]}],
        toolConfig={
            "tools": [{"toolSpec": {"name": tool_name, "description": f"Submit the {tool_name[7:]}.", "inputSchema": {"json": schema}}}],
            "toolChoice": {"tool": {"name": tool_name}},
        },
        inferenceConfig=inference,
    )


def converse_tool(system: str, user: str, tool_name: str, schema: dict[str, Any], client: Any = None) -> dict[str, Any]:
    model_id = os.environ.get("BEDROCK_MODEL_ID")
    if not model_id:
        raise RuntimeError("BEDROCK_MODEL_ID is not set")
    client = client or _bedrock_client()
    try:
        resp = client.converse(**_request(system, user, tool_name, schema, model_id, True))
    except Exception as e:  # noqa: BLE001
        # Some models reject an explicit temperature; retry once without it. Anything else propagates.
        if "temperature" in str(e).lower() and "validation" in str(e).lower():
            resp = client.converse(**_request(system, user, tool_name, schema, model_id, False))
        else:
            raise
    # A truncated tool call can still parse into a *partial* dict (e.g. tripwires silently missing). Refuse it.
    if resp.get("stopReason") != "tool_use":
        raise RuntimeError(f"Bedrock stopped with {resp.get('stopReason')!r}; refusing a possibly truncated answer")
    for block in resp["output"]["message"]["content"]:
        if "toolUse" in block:
            return block["toolUse"]["input"]
    raise RuntimeError("Bedrock response contained no tool use")


# ------------------------------------------------------------------------------ circuit breaker
# After N consecutive Bedrock failures, stop trying for a cool-down so every voice turn doesn't pay the
# full timeout while Bedrock is down. In 'auto' mode we use the offline parser meanwhile; in 'bedrock' mode we fail fast.

class _Breaker:
    def __init__(self):
        self._lock = threading.Lock()
        self.fails, self.open_until = 0, 0.0

    @staticmethod
    def _limits() -> tuple[int, float]:
        return int(os.environ.get("TRIPWIRE_BREAKER_FAILS", "3")), float(os.environ.get("TRIPWIRE_BREAKER_COOLDOWN", "60"))

    def is_open(self) -> bool:
        with self._lock:
            return time.monotonic() < self.open_until

    def success(self) -> None:
        with self._lock:
            self.fails, self.open_until = 0, 0.0

    def failure(self) -> None:
        threshold, cooldown = self._limits()
        with self._lock:
            self.fails += 1
            if self.fails >= threshold:
                self.open_until = time.monotonic() + cooldown

    def reset(self) -> None:
        self.success()


BREAKER = _Breaker()


def _finite(x: Any) -> float | None:
    try:
        f = float(x)
    except (TypeError, ValueError):
        return None
    return f if math.isfinite(f) else None


def _clean_tripwire(t: dict[str, Any]) -> dict[str, Any] | None:
    desc = str(t.get("description", "")).strip()
    if not desc:
        return None
    out: dict[str, Any] = {"description": desc[:160], "metric": None, "op": None, "value": None, "unit": ""}
    value = _finite(t.get("value"))
    if t.get("op") in OPS and value is not None and str(t.get("metric", "")).strip():
        out.update(metric=str(t["metric"]).strip().lower()[:60], op=t["op"], value=value, unit=str(t.get("unit", "")).strip()[:20])
    return out


def validate_extraction(raw: dict[str, Any]) -> dict[str, Any]:
    choice = str(raw.get("choice", "")).strip()
    if not choice:
        raise ValueError("no decision found")
    tws = [c for c in (_clean_tripwire(t) for t in raw.get("tripwires", []) if isinstance(t, dict)) if c]
    days = _finite(raw.get("revisit_in_days"))
    return {
        "title": (str(raw.get("title", "")).strip() or choice)[:60].lower(),
        "choice": choice[:120],
        "options": [str(o).strip()[:80] for o in raw.get("options", []) if str(o).strip()][:6],
        "reason": str(raw.get("reason", "")).strip()[:240],
        "tripwires": tws[:8],
        "revisit_in_days": max(1, min(int(days), 3650)) if days else None,
    }


def validate_assessment(raw: dict[str, Any], known_ids: set[str]) -> list[dict[str, Any]]:
    out = []
    for o in raw.get("observations", []):
        if not isinstance(o, dict) or o.get("tripwire_id") not in known_ids:
            continue  # ignore hallucinated ids
        out.append({
            "tripwire_id": o["tripwire_id"],
            "observed_value": _finite(o.get("observed_value")) if not isinstance(o.get("observed_value"), bool) else None,
            "verdict": o.get("verdict") if o.get("verdict") in ("tripped", "not_tripped", "unsure") else "unsure",
            "note": str(o.get("note", ""))[:200],
        })
    return out


def assess_prompt(fact: str, items: list[dict[str, Any]]) -> str:
    lines = []
    for it in items:
        spec = f" [numeric: {it['metric']} {it['op']} {it['value']:g} {it.get('unit', '')}]" if it.get("metric") else ""
        lines.append(f"- id={it['id']} | decision: {it['decision']} | tripwire: {it['description']}{spec}")
    return "New fact: " + fact + "\n\nArmed tripwires:\n" + "\n".join(lines)


# ------------------------------------------------------------------------------ offline fallback

_DIR = r"(above|over|exceeds?|more than|higher than|at least|below|under|less than|fewer than|lower than)"
_NUM = r"(\$?\d[\d,]*(?:\.\d+)?|once|twice|one|two|three|four|five|ten|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred)"
_FILLER = {"the", "our", "my", "a", "an", "it", "goes", "gets", "go", "get", "rises", "climbs", "drops", "falls",
          "hits", "hit", "is", "are", "reaches", "reach", "ever", "ends", "up", "we", "i", "you", "they", "he", "she", "us"}
_UP = {"above", "over", "exceed", "exceeds", "more than", "higher than"}


def _num(s: str) -> float:
    s = s.replace("$", "").replace(",", "")
    return float(NUMWORDS[s]) if s in NUMWORDS else float(s)


def _clean_text(text: str) -> str:
    """Normalise what speech-to-text / phone keyboards produce: curly quotes, odd whitespace, absurd length."""
    t = text.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    return " ".join(t.split())[:MAX_TEXT]


def extract_offline(text: str) -> dict[str, Any]:
    low = _clean_text(text).lower()
    cm = re.search(
        r"(?:going with|go with|picking|pick|choosing|chose|decided on|decided to (?:get|buy|take|use)|"
        r"getting|buying|taking|we(?:'ll| will) (?:get|take|use))\s+(.+?)(?=\s+(?:because|since|but|we'd|we would|i'd|i would|if|unless)\b|[.,;]|$)", low)
    choice = cm.group(1).strip() if cm else ""
    rm = re.search(r"(?:because|since)\s+(.+?)(?=[.;,]|\s+(?:we'd|we would|i'd|i would)\b|$)", low)
    tws = []
    for m in re.finditer(_DIR + r"\s+" + _NUM + r"(?: times)?(?:\s+(?:a|per|each)\s+(month|week|year|day))?", low):
        seg = re.split(r"[,;.]|\b(?:if|or|and|unless|when|that|switch|change|than)\b", low[:m.start()])[-1]
        metric = " ".join(w for w in seg.split() if w not in _FILLER).strip()
        if not metric:
            continue
        op = ">" if m.group(1) in _UP else "<"
        if m.group(1) == "at least":
            op = ">="
        value = _num(m.group(2))
        unit = f"per {m.group(3)}" if m.group(3) else ("usd" if "$" in m.group(2) else "")
        shown = f"${value:g}" if "$" in m.group(2) else f"{value:g}"
        tws.append({"description": f"{metric} {m.group(1)} {shown}" + (f" {unit}" if unit.startswith("per") else ""),
                    "metric": metric + (f" {unit}" if unit.startswith("per") else ""), "op": op, "value": value, "unit": unit})
    rv = re.search(r"(?:revisit|check back|check in|review|remind (?:us|me))\b[^.]*?\b(?:in|after)\s+(\d+|a|one|two|three|four|six|twelve)\s+(day|week|month|year)s?", low)
    days = None
    if rv:
        n = float(NUMWORDS.get(rv.group(1), rv.group(1)) if not rv.group(1).isdigit() else rv.group(1))
        days = int(n * {"day": 1, "week": 7, "month": 30, "year": 365}[rv.group(2)])
    return validate_extraction({"title": " ".join([w for w in choice.split() if w not in {"the", "a", "an", "our", "my", "some"}][-2:]) if choice else "", "choice": choice,
                                "reason": rm.group(1) if rm else "", "tripwires": tws, "revisit_in_days": days})


def _last_number(text: str) -> float | None:
    found = re.findall(r"\$?\d[\d,]*(?:\.\d+)?|\b(?:once|twice|one|two|three|four|five|six|seven|eight|nine|ten|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred)\b", _clean_text(text).lower())
    return _num(found[-1]) if found else None


def assess_offline(fact: str, items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    ftok, value, out = tokens(_clean_text(fact)), _last_number(fact), []
    numeric = [it for it in items if it.get("metric") and tokens(it["metric"]) and tokens(it["metric"]) <= ftok]
    for it in items:
        about_decision = bool(tokens(it["decision"]) & ftok)
        if it in numeric and value is not None and (about_decision or len(numeric) == 1):
            out.append({"tripwire_id": it["id"], "observed_value": value, "verdict": "unsure", "note": ""})
        elif not it.get("metric"):
            dt = tokens(it["description"])
            if dt and len(dt & ftok) / len(dt) >= 0.6:
                out.append({"tripwire_id": it["id"], "observed_value": None, "verdict": "unsure", "note": "wording overlaps"})
    return out


# ------------------------------------------------------------------------------ public API


def _mode() -> str:
    return os.environ.get("TRIPWIRE_AI", "auto").lower()


def _cap_items(fact: str, items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Bound prompt size: keep the tripwires most related to the fact when there are too many."""
    if len(items) <= MAX_PROMPT_ITEMS:
        return items
    ftok = tokens(fact)
    def rel(it):
        return len(ftok & (tokens(it["decision"]) | tokens(it["description"]) | tokens(it.get("metric") or "")))
    return sorted(items, key=rel, reverse=True)[:MAX_PROMPT_ITEMS]


def _via_bedrock(call):
    """Run a Bedrock call under the circuit breaker."""
    if BREAKER.is_open():
        raise RuntimeError("Bedrock temporarily unavailable (circuit open)")
    try:
        result = call()
    except Exception:
        BREAKER.failure()
        raise
    BREAKER.success()
    return result


def extract_decision(text: str) -> tuple[dict[str, Any], str]:
    text = _clean_text(text)
    if _mode() == "offline":
        return extract_offline(text), "offline"
    try:
        return validate_extraction(_via_bedrock(lambda: converse_tool(EXTRACT_SYSTEM, text, "submit_decision", EXTRACT_SCHEMA))), "bedrock"
    except Exception:
        if _mode() == "bedrock":
            raise
        return extract_offline(text), "offline-fallback"


def assess_fact(fact: str, items: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], str]:
    fact = _clean_text(fact)
    if _mode() == "offline":
        return assess_offline(fact, items), "offline"
    sent = _cap_items(fact, items)
    known = {it["id"] for it in sent}
    try:
        raw = _via_bedrock(lambda: converse_tool(ASSESS_SYSTEM, assess_prompt(fact, sent), "submit_assessment", ASSESS_SCHEMA))
        return validate_assessment(raw, known), "bedrock"
    except Exception:
        if _mode() == "bedrock":
            raise
        return assess_offline(fact, items), "offline-fallback"

In [ ]:
%%writefile tripwire_colab/src/tripwire/service.py
"""Domain logic for Tripwire (transport-agnostic, so it is trivially testable).

Lifecycle of a decision:  active --(tripwire trips / revisit date passes)--> revisit
                          revisit --resolve(keep)--> active | resolve(change) --> changed | resolve(close) --> closed

Safety principles
* The language model may *extract* and *relate*; only this module changes state, and numeric tripwires are
  always compared here, never by the model.
* Tripwires have stable ids, so a verdict computed while someone else edits the decision can never be applied
  to the wrong tripwire.
* Every mutation is an atomic compare-and-swap read-modify-write (``Store.update``): concurrent requests are
  merged, not overwritten, and re-running an already-applied observation is a no-op (idempotent).
"""

from __future__ import annotations

import operator
import os
import re
import uuid
from collections import defaultdict
from datetime import date, datetime, timedelta
from typing import Any, Callable

from . import ai
from .store import ConflictError, Store

KIND = "decision"
OPS = {">": operator.gt, ">=": operator.ge, "<": operator.lt, "<=": operator.le}
NEAR = 0.10  # within 10% of a threshold counts as "getting close"
LIVE = ("active", "revisit")
MAX_DECISIONS, MAX_TRIPWIRES, MAX_HISTORY = 300, 8, 50


def _fmt(v: float) -> str:
    return f"{v:g}"


def _speak_list(items: list[str]) -> str:
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


def _day(d: str) -> str:
    return date.fromisoformat(d).strftime("%B %-d")


def evaluate(tw: dict[str, Any], observed: float) -> bool:
    return OPS[tw["op"]](observed, tw["value"])


def _is_numeric(tw: dict[str, Any]) -> bool:
    return bool(tw.get("metric")) and tw.get("op") in OPS and tw.get("value") is not None


def default_today() -> date:
    """Today in TRIPWIRE_TZ (e.g. 'America/Los_Angeles'); falls back to the server's local date."""
    tz = os.environ.get("TRIPWIRE_TZ")
    if tz:
        try:
            from zoneinfo import ZoneInfo

            return datetime.now(ZoneInfo(tz)).date()
        except Exception:  # noqa: BLE001 - bad tz name must not break every request
            pass
    return date.today()


def _new_tripwire(t: dict[str, Any]) -> dict[str, Any]:
    return {**t, "id": uuid.uuid4().hex[:8], "state": "armed"}


class Tripwire:
    def __init__(self, store: Store, household: str = "home", today: Callable[[], date] = default_today):
        self.s, self.h, self._today = store, household, today

    # ------------------------------------------------------------------ helpers
    def _all(self) -> list[dict[str, Any]]:
        return [{**d, "key": d["_key"]} for d in self.s.list(self.h, KIND)]

    def _event(self, d: dict[str, Any], text: str) -> None:
        hist = d.setdefault("history", [])
        hist.append({"on": self._today().isoformat(), "text": text[:300]})
        if len(hist) > MAX_HISTORY:  # keep the original decision + the most recent events; documents stay small
            d["history"] = [hist[0]] + hist[-(MAX_HISTORY - 1):]

    def _find(self, topic: str, statuses: tuple[str, ...] | None = None):
        """Resolve a spoken topic ('the internet thing') to one decision, or return a speakable error."""
        topic = ai._clean_text(topic or "")
        cands = [d for d in self._all() if statuses is None or d["status"] in statuses]
        want = ai.tokens(topic)
        scored = []
        for d in cands:
            have = ai.tokens(" ".join([d["title"], d["choice"], d.get("reason", ""), " ".join(d.get("options", []))]))
            scored.append((len(want & have), d))
        scored.sort(key=lambda x: -x[0])
        if not scored or scored[0][0] == 0:
            return None, {"ok": False, "spoken": f"I don't have a decision about {topic or 'that'}."}
        if len(scored) > 1 and scored[1][0] == scored[0][0]:
            names = _speak_list([s[1]["title"] for s in scored if s[0] == scored[0][0]])
            return None, {"ok": False, "ambiguous": True, "spoken": f"Which one do you mean: {names}?"}
        return scored[0][1], None

    def _edit(self, key: str, statuses: tuple[str, ...] | None, fn: Callable[[dict[str, Any]], tuple[dict[str, Any] | None, dict[str, Any]]]) -> dict[str, Any]:
        """Atomically edit one decision. ``fn(doc) -> (new_doc | None, response)`` runs against the *freshest* copy
        (and again on conflict), so preconditions are re-checked on current data."""
        box: dict[str, Any] = {}

        def wrapper(doc):
            if doc is None or (statuses and doc["status"] not in statuses):
                box["resp"] = {"ok": False, "spoken": "That decision just changed. Please ask again."}
                return None
            new, box["resp"] = fn(doc)
            return new

        self.s.update(self.h, KIND, key, wrapper)
        return box["resp"]

    def _unique_key(self, title: str, taken: set[str]) -> str:
        base = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")[:40] or "decision"
        key, n = base, 2
        while key in taken:
            key, n = f"{base}-{n}", n + 1
        return key

    def _due_flag(self, d: dict[str, Any]) -> bool:
        return bool(d.get("revisit_on")) and date.fromisoformat(d["revisit_on"]) <= self._today()

    def _due(self, d: dict[str, Any]) -> bool:
        return self._due_flag(d) and d["status"] == "active"

    @staticmethod
    def _public(d: dict[str, Any]) -> dict[str, Any]:
        return {k: v for k, v in d.items() if k not in ("_key",)}

    @staticmethod
    def _bad_number(n: Any, count: int) -> bool:
        return not isinstance(n, int) or isinstance(n, bool) or not 1 <= n <= count

    # ------------------------------------------------------------------ record
    def record_decision(self, what_was_said: str) -> dict[str, Any]:
        text = ai._clean_text(what_was_said or "")
        if len(text) < 8:
            return {"ok": False, "spoken": "I didn't catch that. Try: we're going with X because Y, and we'd change our mind if Z."}
        try:
            ext, engine = ai.extract_decision(text)
        except Exception:  # noqa: BLE001 - speakable, not a stack trace
            return {"ok": False, "spoken": "I couldn't make sense of that decision. Try: we're going with X because Y, and we'd change our mind if Z."}
        existing = self._all()
        # Idempotency: a client that times out and retries must not create a duplicate (which would make every
        # later "the internet one" ambiguous). Same live decision recorded today with the same content = same request.
        sig = (ai.tokens(ext["choice"]), tuple(sorted(t["description"].lower() for t in ext["tripwires"])))
        for d in existing:
            if (d["status"] in LIVE and d["created_on"] == self._today().isoformat()
                    and (ai.tokens(d["choice"]), tuple(sorted(t["description"].lower() for t in d["tripwires"]))) == sig):
                return {"ok": True, "engine": engine, "duplicate": True, "decision": d,
                        "spoken": f"I already have that one: {d['choice']}. Nothing changed."}
        if len(existing) >= MAX_DECISIONS:
            return {"ok": False, "spoken": "I'm tracking a lot of decisions already. Close a few and I'll make room."}
        today = self._today()
        doc = {
            "title": ext["title"], "choice": ext["choice"], "options": ext["options"], "reason": ext["reason"],
            "status": "active", "created_on": today.isoformat(),
            "tripwires": [_new_tripwire(t) for t in ext["tripwires"][:MAX_TRIPWIRES]],
            "revisit_on": (today + timedelta(days=ext["revisit_in_days"])).isoformat() if ext["revisit_in_days"] else None,
            "history": [],
        }
        self._event(doc, f"Decided: {doc['choice']}" + (f" because {doc['reason']}" if doc["reason"] else ""))
        taken = {d["key"] for d in existing}
        for _ in range(25):  # create-only write (expected_ver=0); on a name race, take the next free key
            key = self._unique_key(ext["title"], taken)
            try:
                self.s.put(self.h, KIND, key, doc, expected_ver=0)
                break
            except ConflictError:
                taken = {d["key"] for d in self._all()}
        else:
            raise ConflictError("could not allocate a decision key")
        spoken = f"Got it. You chose {doc['choice']}" + (f", because {doc['reason']}." if doc["reason"] else ".")
        if doc["tripwires"]:
            numbered = [f"{i}, {t['description']}" for i, t in enumerate(doc["tripwires"], 1)]
            spoken += f" I'm watching for: {'; '.join(numbered)}."
        else:
            spoken += " I didn't hear what would change your mind. Tell me and I'll add a tripwire."
        if doc["revisit_on"]:
            spoken += f" I'll flag it on {_day(doc['revisit_on'])}."
        return {"ok": True, "engine": engine, "decision": {**doc, "key": key}, "spoken": spoken}

    def add_tripwire(self, topic: str, condition: str) -> dict[str, Any]:
        condition = ai._clean_text(condition or "")
        if len(condition) < 3:
            return {"ok": False, "spoken": "What condition should I watch for?"}
        d, err = self._find(topic, LIVE)
        if err:
            return err
        engine, found = "none", []
        try:  # reuse the decision extractor, framing the condition so it parses like a spoken decision
            ext, engine = ai.extract_decision(f"We're going with {d['choice']}. We'd change our mind if {condition}")
            found = ext["tripwires"]
        except Exception:  # noqa: BLE001 - fall back to a plain, human-confirmed tripwire
            pass
        blank = {"metric": None, "op": None, "value": None, "unit": ""}
        new = [_new_tripwire(t) for t in found] or [_new_tripwire({"description": condition[:160], **blank})]

        def fn(doc):
            have = {t["description"].lower() for t in doc["tripwires"]}
            new_ = [t for t in new if t["description"].lower() not in have]
            if not new_:  # retry-safe: already watching exactly this
                return None, {"ok": True, "engine": engine, "spoken": f"I'm already watching for {new[0]['description']}."}
            room = MAX_TRIPWIRES - len(doc["tripwires"])
            if room <= 0:
                return None, {"ok": False, "spoken": f"{doc['title'].capitalize()} already has {MAX_TRIPWIRES} tripwires. Remove one first."}
            doc["tripwires"].extend(new_[:room])
            self._event(doc, f"Added tripwire: {new_[0]['description']}")
            return doc, {"ok": True, "engine": engine, "spoken": f"Added. For {doc['title']} I'm now watching {len(doc['tripwires'])} things, including {new_[0]['description']}."}

        return self._edit(d["key"], LIVE, fn)

    def remove_tripwire(self, topic: str, number: int) -> dict[str, Any]:
        d, err = self._find(topic, LIVE)
        if err:
            return err

        def fn(doc):
            if self._bad_number(number, len(doc["tripwires"])):
                return None, {"ok": False, "spoken": f"{doc['title'].capitalize()} has {len(doc['tripwires'])} tripwires. Which number?"}
            gone = doc["tripwires"].pop(number - 1)
            self._event(doc, f"Removed tripwire: {gone['description']}")
            if doc["status"] == "revisit" and not any(t["state"] in ("tripped", "pending") for t in doc["tripwires"]) and not self._due_flag(doc):
                doc["status"] = "active"
            return doc, {"ok": True, "spoken": f"Removed {gone['description']}."}

        return self._edit(d["key"], LIVE, fn)

    # ------------------------------------------------------------------ the core: reporting facts
    def _apply_observations(self, doc: dict[str, Any], observations: list[dict[str, Any]], statement: str):
        """Pure function of (doc, observations): returns (new_doc | None if unchanged, result lists).
        Observations name tripwires by stable id; ones that no longer exist, or are no longer armed (already
        handled by a concurrent/previous report), are skipped, so applying twice is a no-op."""
        by_id = {t["id"]: t for t in doc["tripwires"] if "id" in t}
        res: dict[str, list] = {"tripped": [], "maybe": [], "near": []}
        changed = False
        for o in observations:
            tw = by_id.get(o["tripwire_id"].rsplit("#", 1)[-1])
            if tw is None or tw["state"] != "armed":
                continue
            val = o["observed_value"]
            if _is_numeric(tw) and val is not None:
                # server-side truth: the model's verdict is ignored for numeric tripwires
                if evaluate(tw, val):
                    tw.update(state="tripped", observed=val, on=self._today().isoformat(), statement=statement[:200])
                    self._event(doc, f"Tripped: {tw['description']} (observed {_fmt(val)})")
                    res["tripped"].append((doc["title"], tw["description"], val))
                    changed = True
                elif tw["value"] and abs(val - tw["value"]) / abs(tw["value"]) <= NEAR:
                    res["near"].append((doc["title"], tw["description"], val, tw["value"]))
            elif not _is_numeric(tw) and o["verdict"] in ("tripped", "unsure"):
                tw.update(state="pending", statement=statement[:200], note=o.get("note", ""))
                res["maybe"].append((doc["title"], tw["description"]))
                changed = True
        if any(t["state"] == "tripped" for t in doc["tripwires"]) and doc["status"] == "active":
            doc["status"] = "revisit"
            changed = True
        return (doc if changed else None), res

    def report_fact(self, statement: str) -> dict[str, Any]:
        statement = ai._clean_text(statement or "")
        if len(statement) < 3:
            return {"ok": False, "spoken": "What did you want me to note?"}
        decisions = [d for d in self._all() if d["status"] in LIVE]
        items = [{"id": f"{d['key']}#{t['id']}", "decision": d["title"], **{k: t.get(k) for k in ("description", "metric", "op", "value", "unit")}}
                 for d in decisions for t in d["tripwires"] if t["state"] == "armed" and "id" in t]
        if not items:
            return {"ok": True, "tripped": [], "spoken": "Noted, but I'm not watching anything that this relates to."}
        try:
            obs, engine = ai.assess_fact(statement, items)
        except Exception:  # noqa: BLE001
            return {"ok": False, "spoken": "I couldn't check that against your tripwires just now. Please try again."}
        grouped: dict[str, list] = defaultdict(list)
        for o in obs:
            grouped[o["tripwire_id"].rsplit("#", 1)[0]].append(o)
        agg: dict[str, list] = {"tripped": [], "maybe": [], "near": []}
        for key, group in grouped.items():
            box: dict[str, Any] = {}

            def fn(doc, group=group, box=box):
                if doc is None or doc["status"] not in LIVE:
                    box["res"] = None
                    return None
                new, box["res"] = self._apply_observations(doc, group, statement)
                return new

            self.s.update(self.h, KIND, key, fn)
            for k, v in (box.get("res") or {}).items():
                agg[k].extend(v)
        parts = []
        for title, desc, val in agg["tripped"]:
            parts.append(f"That trips your {title} decision: {desc}, and it's now {_fmt(val)}.")
        for title, desc in agg["maybe"]:
            parts.append(f"That might trip your {title} tripwire, {desc}. Did it?")
        for title, desc, val, limit in agg["near"]:
            parts.append(f"Heads up: {title} is at {_fmt(val)}, close to your limit of {_fmt(limit)}.")
        if agg["tripped"]:
            parts.append("Want to keep it, change it, or adjust the limit?")
        return {"ok": True, "engine": engine, "tripped": [t[0] for t in agg["tripped"]],
                "needs_confirmation": [m[0] for m in agg["maybe"]], "close_to_limit": [n[0] for n in agg["near"]],
                "spoken": " ".join(parts) or "Noted. Nothing tripped."}

    def confirm_tripwire(self, topic: str, number: int, tripped: bool) -> dict[str, Any]:
        d, err = self._find(topic, LIVE)
        if err:
            return err

        def fn(doc):
            if self._bad_number(number, len(doc["tripwires"])):
                return None, {"ok": False, "spoken": f"{doc['title'].capitalize()} has {len(doc['tripwires'])} tripwires. Which number?"}
            tw = doc["tripwires"][number - 1]
            if tripped:
                tw.update(state="tripped", on=self._today().isoformat())
                doc["status"] = "revisit"
                self._event(doc, f"Confirmed tripped: {tw['description']}")
                spoken = f"Confirmed. {doc['title'].capitalize()} needs a second look. Keep it, change it, or adjust the limit?"
            else:
                tw["state"] = "armed"
                tw.pop("statement", None)
                self._event(doc, f"Dismissed false alarm: {tw['description']}")
                spoken = "Okay, still watching it."
            return doc, {"ok": True, "spoken": spoken}

        return self._edit(d["key"], LIVE, fn)

    # ------------------------------------------------------------------ revisiting
    def resolve_decision(self, topic: str, outcome: str, note: str | None = None, new_limit: float | None = None,
                         tripwire_number: int | None = None) -> dict[str, Any]:
        if outcome not in ("keep", "change", "close"):
            return {"ok": False, "spoken": "You can keep it, change it, or close it."}
        if new_limit is not None and ai._finite(new_limit) is None:
            return {"ok": False, "spoken": "That doesn't sound like a valid limit."}
        note = ai._clean_text(note)[:200] if note else None
        d, err = self._find(topic, LIVE)
        if err:
            return err

        def fn(doc):
            if outcome == "keep":
                moved = []
                if new_limit is not None:
                    tws = doc["tripwires"]
                    targets = [i for i, t in enumerate(tws, 1) if t["state"] in ("tripped", "pending") and t.get("value") is not None]
                    if tripwire_number is not None:
                        if self._bad_number(tripwire_number, len(tws)) or tws[tripwire_number - 1].get("value") is None:
                            return None, {"ok": False, "spoken": "That tripwire doesn't have a numeric limit I can move."}
                        targets = [tripwire_number]
                    if len(targets) != 1:  # never guess which limit to move
                        opts = "; ".join(f"{i}, {tws[i - 1]['description']}" for i in targets) or "none of them has a numeric limit"
                        return None, {"ok": False, "spoken": f"Which limit should I move? {opts}."}
                    tw = tws[targets[0] - 1]
                    tw["value"] = float(new_limit)
                    tw["description"] = re.sub(r"\$?\d[\d,]*(?:\.\d+)?", ("$" if "$" in tw["description"] else "") + _fmt(float(new_limit)), tw["description"], count=1)
                    moved.append(tw["description"])
                for tw in doc["tripwires"]:
                    if tw["state"] in ("tripped", "pending"):
                        tw["state"] = "armed"
                        for k in ("observed", "on", "statement", "note"):
                            tw.pop(k, None)
                doc["status"] = "active"
                if doc.get("revisit_on") and self._due_flag(doc):
                    doc["revisit_on"] = None
                self._event(doc, "Kept" + (f": {note}" if note else "") + (f"; limit moved ({moved[0]})" if moved else ""))
                spoken = f"Kept {doc['choice']}." + (f" New limit: {moved[0]}." if moved else " I'm watching again.")
            elif outcome == "change":
                doc["status"] = "changed"
                self._event(doc, "Changed" + (f": {note}" if note else ""))
                spoken = f"Okay, I've marked {doc['title']} as changed" + (f" to {note}." if note else ". Record the new decision whenever you're ready.")
            else:
                doc["status"] = "closed"
                self._event(doc, "Closed" + (f": {note}" if note else ""))
                spoken = f"Closed {doc['title']}. I'll stop watching it."
            return doc, {"ok": True, "status": doc["status"], "spoken": spoken}

        return self._edit(d["key"], LIVE, fn)

    # ------------------------------------------------------------------ reading back
    def whats_on_watch(self) -> dict[str, Any]:
        live = [d for d in self._all() if d["status"] in LIVE]
        if not live:
            return {"ok": True, "decisions": [], "spoken": "You're not watching any decisions yet."}
        attention, calm = [], []
        for d in live:
            if d["status"] == "revisit":
                trips = [t for t in d["tripwires"] if t["state"] in ("tripped", "pending")]
                attention.append((d["title"], f"{d['title']}, because {trips[0]['description']}" if trips else f"{d['title']}, it's time to revisit"))
            elif self._due(d):
                attention.append((d["title"], f"{d['title']}, its check-in date has arrived"))
            else:
                calm.append(d["title"])
        spoken = f"You're watching {len(live)} decision{'s' if len(live) != 1 else ''}."
        if attention:
            spoken += f" Needs attention: {_speak_list([a[1] for a in attention])}."
        if calm:
            spoken += f" All quiet on {_speak_list(calm)}."
        return {"ok": True, "needs_attention": [a[0] for a in attention], "decisions": [self._public(d) for d in live], "spoken": spoken}

    def why_did_we(self, topic: str) -> dict[str, Any]:
        d, err = self._find(topic)
        if err:
            return err
        spoken = f"On {_day(d['created_on'])} you chose {d['choice']}"
        spoken += f" because {d['reason']}." if d.get("reason") else "."
        if d.get("options"):
            spoken += f" You also considered {_speak_list(d['options'])}."
        if d["tripwires"]:
            spoken += f" You said you'd reconsider if {_speak_list([t['description'] for t in d['tripwires']])}."
        later = d["history"][1:]
        if later:
            spoken += " Since then: " + "; ".join(f"{_day(h['on'])}, {h['text'].lower()}" for h in later[-3:]) + "."
        spoken += f" Status: {d['status']}."
        return {"ok": True, "decision": self._public(d), "spoken": spoken}

In [ ]:
%%writefile tripwire_colab/src/tripwire/simulator.py
"""Simulated Alexa+ experience.

Real Alexa+ routes what a person says to MCP tools using its own AI. When Alexa+ access is unavailable, the hackathon
rules allow a *simulated* Alexa+ experience. This module is that simulation's routing layer: it maps a spoken sentence to
the **same service calls the MCP tools make** (record_decision, report_fact, ...), using simple, inspectable rules.

It is a stand-in for Alexa+'s routing, NOT for Tripwire's logic: every state change still goes through ``Tripwire``.
Enable the web UI with ``TRIPWIRE_SIM=1`` and open ``/sim``.
"""

from __future__ import annotations

import re
import threading
from datetime import date, timedelta
from typing import Any

from . import ai
from .service import Tripwire, default_today

_WAKE = re.compile(r"^(?:(?:hey |ok |okay )?alexa)[,.!:]?\s+|^(?:(?:tell|ask) tripwire(?: to| that)?[,:]?\s+)", re.I)
_DECISION = re.compile(
    r"\b(?:we|i)(?:'re|'m| are| am| will|'ll)?\s+(?:going with|going to (?:get|buy|take|use)|getting|buying|picking|choosing|"
    r"chose|picked|decided on|decided to (?:get|buy|take|use)|will (?:get|take|use))\b", re.I)
_WHY = re.compile(r"^(?:why (?:did|do|would) (?:we|i)|why'd (?:we|i)|remind me why (?:we|i))\s*(.*)$", re.I)
_WATCH = re.compile(r"what (?:should i|do i need to|do we need to) look at|what'?s on (?:the )?watch|what are you watching|"
                    r"what needs (?:my |our )?attention|anything (?:i|we) should (?:know|look at)", re.I)
_RESOLVE = re.compile(r"^(keep|change|close|drop|stop watching)\b\s*(.*)$", re.I)
_LIMIT = re.compile(r"(?:move|raise|lower|set|change|adjust|bump)\b.*?\blimit\b.*?\bto\s+\$?(\d[\d,]*(?:\.\d+)?)", re.I)


class Simulator:
    """Holds the simulated clock (so the UI can 'skip days') and routes sentences to service calls."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self.offset_days = 0

    def today(self) -> date:
        return default_today() + timedelta(days=self.offset_days)

    def advance(self, days: int) -> int:
        with self._lock:
            self.offset_days = max(0, min(3650, self.offset_days + max(0, min(int(days), 365))))
            return self.offset_days

    # ------------------------------------------------------------------ routing
    def route(self, svc: Tripwire, utterance: str) -> tuple[str, dict[str, Any]]:
        text = ai._clean_text(utterance)
        text = _WAKE.sub("", text).strip()
        m = _WHY.match(text)
        if m:
            return "why_did_we", {"topic": (m.group(1) or text).strip(" ?.")}
        if _WATCH.search(text):
            return "whats_on_watch", {}
        m = _RESOLVE.match(text)
        if m and not _DECISION.search(text):
            return self._resolve_args(svc, m.group(1).lower(), m.group(2), text)
        if _DECISION.search(text):
            return "record_decision", {"what_was_said": text}
        return "report_fact", {"statement": text}

    @staticmethod
    def _resolve_args(svc: Tripwire, verb: str, rest: str, full: str) -> tuple[str, dict[str, Any]]:
        outcome = {"keep": "keep", "change": "change", "close": "close", "drop": "close", "stop watching": "close"}[verb]
        args: dict[str, Any] = {"outcome": outcome}
        lim = _LIMIT.search(full)
        if lim and outcome == "keep":
            args["new_limit"] = float(lim.group(1).replace(",", ""))
        note = re.sub(r"\b(it|that|this|one|but|and|please)\b", "", _LIMIT.sub("", rest), flags=re.I).strip(" ,.;")
        live = [d for d in svc._all() if d["status"] in ("active", "revisit")]
        flagged = [d for d in live if d["status"] == "revisit"]
        named = [d for d in live if ai.tokens(note) & ai.tokens(d["title"] + " " + d["choice"])]
        pool = named or flagged
        if len(pool) == 1:
            args["topic"] = pool[0]["title"]
        elif len(pool) > 1:  # "keep it" with several candidates: ask, never guess
            args["ambiguous"] = [d["title"] for d in pool]
            args["topic"] = ""
        else:
            args["topic"] = note or ""
        if outcome == "change" and note and not named:
            args["note"] = note
        elif outcome == "keep" and note and not named:
            args["note"] = note
        return "resolve_decision", args

    def say(self, svc: Tripwire, utterance: str) -> dict[str, Any]:
        tool, args = self.route(svc, utterance)
        fn = getattr(svc, tool)
        if tool == "resolve_decision":
            if args.get("ambiguous"):
                names = " or ".join(args["ambiguous"])
                result = {"ok": False, "ambiguous": True, "spoken": f"Which one do you mean: {names}?"}
            elif not args["topic"]:
                result = {"ok": False, "spoken": "Nothing needs a decision right now. Tell me which one you mean."}
            else:
                result = fn(args["topic"], args["outcome"], args.get("note"), args.get("new_limit"), None)
        else:
            result = fn(**args)
        return {"tool": tool, "args": args, "result": result, "spoken": result["spoken"]}

    def state(self, svc: Tripwire) -> dict[str, Any]:
        out = []
        for d in svc._all():
            if d["status"] in ("closed",):
                continue
            out.append({"title": d["title"], "status": d["status"], "choice": d["choice"], "tripwires": [
                {"d": t["description"], "op": t.get("op"), "v": t.get("value"), "state": t["state"], "obs": t.get("observed")}
                for t in d["tripwires"]]})
        return {"date": svc._today().strftime("%b %-d, %Y"), "decisions": out, "skipped_days": self.offset_days}

In [ ]:
%%writefile tripwire_colab/src/tripwire/simhtml.py
"""HTML for the simulated Alexa+ experience (served at /sim when TRIPWIRE_SIM=1). All dynamic text uses textContent."""

SIM_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Tripwire: simulated Alexa+</title>
<style>
:root{--bg:#f6f5f1;--card:#fff;--ink:#1c1b19;--mute:#6b6860;--line:#e3e0d8;--acc:#0b6bcb;--ok:#2f7d4f;--warn:#b7791f;--bad:#c0392b;--bub:#eaf2fc;--bar:#e9e6de;color-scheme:light}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#15150f;--card:#1f1f18;--ink:#f1efe8;--mute:#a09d92;--line:#33322a;--acc:#6db1ff;--ok:#6fcf97;--warn:#f2c14e;--bad:#ff7a6b;--bub:#1d2a3a;--bar:#2b2a22;color-scheme:dark}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:16px/1.45 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;padding:env(safe-area-inset-top,0) 0 env(safe-area-inset-bottom,0)}
.wrap{max-width:1100px;margin:0 auto;padding:16px}
.ban{background:color-mix(in srgb,var(--warn) 18%,var(--card));border:1px solid var(--warn);border-radius:10px;padding:8px 12px;font-size:13px;margin-bottom:12px}
header{display:flex;flex-wrap:wrap;gap:10px;align-items:center;justify-content:space-between;margin-bottom:12px}
h1{font-size:21px;margin:0}.date{background:var(--card);border:1px solid var(--line);border-radius:999px;padding:6px 14px;font-weight:600}
.grid{display:grid;grid-template-columns:1.1fr .9fr;gap:14px}@media(max-width:820px){.grid{grid-template-columns:1fr}}
.panel{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:14px}
h2{font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:var(--mute);margin:0 0 10px}
#log{height:340px;overflow:auto;padding-right:4px}
.b{border-radius:14px;padding:9px 13px;margin:0 0 9px;max-width:92%;white-space:pre-wrap}.me{background:var(--bub);margin-left:auto}.al{border:1px solid var(--line)}
.who{font-size:11px;font-weight:700;color:var(--mute)}.tool{font:12px ui-monospace,Menlo,Consolas,monospace;color:var(--acc);margin:-4px 0 9px}
.in{display:flex;gap:8px;margin-top:10px}input[type=text]{flex:1;font:inherit;padding:10px 12px;border-radius:10px;border:1px solid var(--line);background:var(--bg);color:var(--ink)}
button{font:inherit;font-weight:600;background:var(--card);color:var(--ink);border:1px solid var(--line);border-radius:10px;padding:9px 14px;cursor:pointer}button.p{background:var(--acc);color:#fff;border-color:var(--acc)}
button:focus-visible,input:focus-visible{outline:2px solid var(--acc);outline-offset:2px}
.chips{display:flex;flex-wrap:wrap;gap:6px;margin-top:10px}.chips button{font-weight:500;font-size:13px;padding:5px 10px}
.opt{font-size:13px;color:var(--mute);margin-top:8px;display:flex;gap:14px;align-items:center;flex-wrap:wrap}
.dec{border:1px solid var(--line);border-radius:12px;padding:10px;margin-bottom:10px}.dec.revisit{border-color:var(--bad);box-shadow:0 0 0 3px color-mix(in srgb,var(--bad) 18%,transparent)}
.dh{display:flex;justify-content:space-between;font-weight:600;text-transform:capitalize}.badge{font-size:11px;font-weight:700;text-transform:uppercase;border:1px solid currentColor;border-radius:99px;padding:1px 8px}
.active{color:var(--ok)}.revisit{color:var(--bad)}.ch{color:var(--mute);font-size:13px}.tw{margin-top:7px;font-size:14px}.row{display:flex;justify-content:space-between;gap:8px}
.bar{height:8px;border-radius:99px;background:var(--bar);margin-top:4px;overflow:hidden}.fill{height:100%;background:var(--ok);transition:width .6s}.fill.tripped{background:var(--bad)}
.st{font-size:12px;font-weight:700;text-transform:uppercase}.st.armed{color:var(--ok)}.st.tripped,.st.pending{color:var(--bad)}
</style></head><body><div class="wrap">
<div class="ban"><b>Simulated Alexa+ experience.</b> Not a real Alexa device. Simple routing rules stand in for Alexa+'s AI and call the same
Tripwire logic the MCP server exposes; every answer and every saved change comes from the real Tripwire code.</div>
<header><h1>🪤 Tripwire <span class="ch">· simulated Alexa+</span></h1>
<div><span class="date" id="date"></span> <button id="skip" title="Simulate time passing">Skip 7 days ⏩</button></div></header>
<div class="grid">
<section class="panel"><h2>Talk to Alexa+</h2><div id="log" aria-live="polite"></div>
<div class="in"><input id="q" type="text" placeholder="Type, or press 🎤 and speak…" autocomplete="off" aria-label="What you say to Alexa"><button id="mic" title="Speak" aria-label="Speak">🎤</button><button id="go" class="p">Send</button></div>
<div class="opt"><label><input type="checkbox" id="tts"> Speak answers aloud</label><span id="hint"></span></div>
<div class="chips" id="chips"></div></section>
<section class="panel"><h2>What Tripwire remembers</h2><div id="mem"></div></section></div></div>
<script>
const $=id=>document.getElementById(id),fix=s=>String(s).replace(/(\w)'(\w)/g,"$1\u2019$2");
const EX=["Alexa, we're going with the 200 megabit internet plan because it's cheapest. We'd switch if the price goes above $50 or if outages hit more than twice a month. Check back in three months.",
"We're getting the grey sofa because it's comfy. We'd return it if the price drops below $300.",
"The provider emailed that our internet price goes to $48 next month.","The provider just raised our internet price to $55.",
"What should I look at?","Why did we pick that internet plan?","Keep it, but move the price limit to $60.","The sofa is on sale. The price dropped to $250."];
function add(cls,who,text,tool){const d=document.createElement("div");d.className="b "+cls;const w=document.createElement("div");w.className="who";w.textContent=who;
 const t=document.createElement("div");t.textContent=fix(text);d.append(w,t);$("log").append(d);
 if(tool){const x=document.createElement("div");x.className="tool";x.textContent="→ tool: "+tool;$("log").append(x)}$("log").scrollTop=1e9}
function gauge(t){const r=document.createElement("div");r.className="tw";
 const top=document.createElement("div");top.className="row";const a=document.createElement("span");a.textContent=fix(t.d);const s=document.createElement("span");s.className="st "+t.state;s.textContent=t.state;top.append(a,s);r.append(top);
 if(t.v!=null){const b=document.createElement("div");b.className="bar";const f=document.createElement("div");f.className="fill "+(t.state==="tripped"?"tripped":"");
  let p=0;if(t.obs!=null){p=t.op.startsWith(">")?t.obs/t.v:t.v/t.obs}f.style.width=Math.max(0,Math.min(1,p))*100+"%";b.append(f);r.append(b);
  const c=document.createElement("div");c.className="ch";c.textContent="limit "+t.v+(t.obs!=null?" · seen "+t.obs:"");r.append(c)}return r}
async function mem(){const s=await (await fetch("/sim/state")).json();$("date").textContent="📅 "+s.date;const m=$("mem");m.replaceChildren();
 if(!s.decisions.length){const e=document.createElement("div");e.className="ch";e.textContent="Nothing on watch yet.";m.append(e)}
 for(const d of s.decisions){const c=document.createElement("div");c.className="dec "+d.status;const h=document.createElement("div");h.className="dh";
  const n=document.createElement("span");n.textContent=d.title;const bd=document.createElement("span");bd.className="badge "+d.status;bd.textContent=d.status==="revisit"?"needs a look":d.status;h.append(n,bd);
  const ch=document.createElement("div");ch.className="ch";ch.textContent="chose: "+fix(d.choice);c.append(h,ch);d.tripwires.forEach(t=>c.append(gauge(t)));m.append(c)}}
async function post(u,b){const r=await fetch(u,{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify(b)});return r.json()}
async function send(text){text=(text||$("q").value).trim();if(!text)return;$("q").value="";add("me","YOU",text);
 try{const r=await post("/sim/say",{utterance:text});add("al","ALEXA+",r.spoken||"Sorry, something went wrong.",r.tool);
  if($("tts").checked&&window.speechSynthesis)speechSynthesis.speak(new SpeechSynthesisUtterance(fix(r.spoken||"")))}catch(e){add("al","ALEXA+","I can't reach Tripwire right now.")}await mem()}
$("go").onclick=()=>send();$("q").addEventListener("keydown",e=>{if(e.key==="Enter")send()});
$("skip").onclick=async()=>{await post("/sim/advance",{days:7});await mem()};
EX.forEach(x=>{const b=document.createElement("button");b.textContent=fix(x.length>46?x.slice(0,44)+"…":x);b.title=x;b.onclick=()=>send(x);$("chips").append(b)});
const SR=window.SpeechRecognition||window.webkitSpeechRecognition;
if(SR){const r=new SR();r.lang="en-US";r.onresult=e=>{$("q").value=e.results[0][0].transcript;send()};r.onend=()=>$("mic").textContent="🎤";
 $("mic").onclick=()=>{$("mic").textContent="…";try{r.start()}catch(e){}}}else{$("mic").hidden=true;$("hint").textContent="(voice input needs Chrome or Edge)"}
mem();
</script></body></html>"""

In [ ]:
%%writefile tripwire_colab/src/tripwire/server.py
"""MCP server (Streamable HTTP) exposing Tripwire to Alexa+ or any MCP host."""

from __future__ import annotations

import hmac
import logging
import os
from typing import Annotated, Any, Literal

import anyio
import anyio.to_thread
from mcp.server.mcpserver import MCPServer
from mcp.server.transport_security import TransportSecuritySettings
from mcp.types import ToolAnnotations
from pydantic import Field
from starlette.responses import JSONResponse, Response

from .service import Tripwire
from .simulator import Simulator
from .store import make_store

log = logging.getLogger("tripwire")
BUSY = "I'm having trouble reaching my memory right now. Please try again in a moment."

INSTRUCTIONS = """\
Tripwire remembers household decisions together with the conditions ('tripwires') that would change them.

- When the user states a decision, call record_decision with their words verbatim; do not paraphrase numbers.
- When the user mentions ANY new fact that might matter ('the provider raised the price to 55', 'we had three
  outages'), call report_fact with their words, even if they didn't ask. This is how tripwires fire.
- 'Why did we choose X?' -> why_did_we. 'What should I look at?' -> whats_on_watch.
- After a tripwire fires, ask whether to keep, change, or close the decision, then call resolve_decision.
Every result has a short `spoken` field that is safe to read aloud verbatim.
"""

TOPIC = Annotated[str, Field(description="Spoken topic, e.g. 'internet plan' or 'the couch'")]


def default_service() -> Tripwire:
    return Tripwire(make_store(), household=os.environ.get("TRIPWIRE_HOUSEHOLD", "home"))


def call(fn, *args) -> dict[str, Any]:
    """Run a service method; any failure (DynamoDB down, timeout, lost race...) becomes a speakable
    sentence for the user and a logged stack trace for the operator. Internals are never leaked to the caller."""
    try:
        return fn(*args)
    except Exception:  # noqa: BLE001
        log.exception("tool call failed: %s", getattr(fn, "__name__", fn))
        return {"ok": False, "error": "temporarily_unavailable", "spoken": BUSY}


def build_server(service: Tripwire | None = None) -> MCPServer:
    svc = service or default_service()
    mcp = MCPServer("Tripwire", instructions=INSTRUCTIONS, version="0.1.0")
    ro = ToolAnnotations(readOnlyHint=True, openWorldHint=False)
    rw = ToolAnnotations(readOnlyHint=False, destructiveHint=False, openWorldHint=False)

    @mcp.tool(annotations=rw)
    def record_decision(what_was_said: Annotated[str, Field(description="The user's own words describing the decision, why, and what would change their mind", max_length=1500)]) -> dict[str, Any]:
        """Remember a decision and the tripwires (conditions that would make the user reconsider). Pass the user's words as spoken."""
        return call(svc.record_decision, what_was_said)

    @mcp.tool(annotations=rw)
    def report_fact(statement: Annotated[str, Field(description="A new fact the user mentioned, as spoken", max_length=500)]) -> dict[str, Any]:
        """Check a new fact against every armed tripwire. Numeric thresholds are compared server-side. Call this whenever the user mentions news that could affect a past decision."""
        return call(svc.report_fact, statement)

    @mcp.tool(annotations=ro)
    def whats_on_watch() -> dict[str, Any]:
        """List live decisions, highlighting any that tripped or whose check-in date has arrived."""
        return call(svc.whats_on_watch)

    @mcp.tool(annotations=ro)
    def why_did_we(topic: TOPIC) -> dict[str, Any]:
        """Recall why a decision was made, what was considered, its tripwires and what has happened since."""
        return call(svc.why_did_we, topic)

    @mcp.tool(annotations=rw)
    def confirm_tripwire(topic: TOPIC, number: Annotated[int, Field(ge=1, le=20)], tripped: bool) -> dict[str, Any]:
        """Answer 'did that trip it?' for a possible match (qualitative tripwires need a human yes/no)."""
        return call(svc.confirm_tripwire, topic, number, tripped)

    @mcp.tool(annotations=rw)
    def resolve_decision(topic: TOPIC, outcome: Literal["keep", "change", "close"], note: str | None = None,
                         new_limit: float | None = None, tripwire_number: Annotated[int | None, Field(ge=1, le=20)] = None) -> dict[str, Any]:
        """Finish a revisit: 'keep' (optionally moving a numeric limit to new_limit), 'change', or 'close'. If several limits tripped, pass tripwire_number."""
        return call(svc.resolve_decision, topic, outcome, note, new_limit, tripwire_number)

    @mcp.tool(annotations=rw)
    def add_tripwire(topic: TOPIC, condition: Annotated[str, Field(description="e.g. 'the price goes above 60'", max_length=300)]) -> dict[str, Any]:
        """Add another condition that would make the user reconsider an existing decision."""
        return call(svc.add_tripwire, topic, condition)

    @mcp.tool(annotations=ToolAnnotations(readOnlyHint=False, destructiveHint=True, openWorldHint=False))
    def remove_tripwire(topic: TOPIC, number: Annotated[int, Field(ge=1, le=20)]) -> dict[str, Any]:
        """Stop watching one tripwire (numbered as read back when the decision was recorded)."""
        return call(svc.remove_tripwire, topic, number)

    return mcp
class BearerAuth:
    """Minimal ASGI guard. If TRIPWIRE_TOKEN is set, every request except /healthz needs it."""

    def __init__(self, app, token: str):
        self.app, self.token = app, token.encode()

    async def __call__(self, scope, receive, send):
        if scope["type"] == "http" and scope["path"] not in ("/healthz", "/readyz"):
            auth = dict(scope["headers"]).get(b"authorization", b"")
            if not (auth.lower().startswith(b"bearer ") and hmac.compare_digest(auth[7:], self.token)):
                resp = JSONResponse({"error": "unauthorized"}, status_code=401, headers={"WWW-Authenticate": "Bearer"})
                return await resp(scope, receive, send)
        return await self.app(scope, receive, send)


def _add_sim_routes(app, svc: Tripwire, sim: Simulator) -> None:
    """Simulated Alexa+ web UI (opt-in via TRIPWIRE_SIM=1; meant for local demos, not public deployments)."""
    from starlette.responses import HTMLResponse

    from .simhtml import SIM_HTML

    async def page(_r) -> Response:
        return HTMLResponse(SIM_HTML)

    async def body(request):
        # Require JSON: browsers can't send it cross-site without a CORS preflight, which blocks drive-by requests.
        if "application/json" not in request.headers.get("content-type", ""):
            return None
        try:
            data = await request.json()
        except Exception:  # noqa: BLE001
            return None
        return data if isinstance(data, dict) else None

    async def say(request) -> Response:
        data = await body(request)
        text = data.get("utterance") if data else None
        if not isinstance(text, str) or not text.strip():
            return JSONResponse({"ok": False, "spoken": "Say something and I'll pass it to Tripwire."}, status_code=400 if data is not None else 415)
        out = await anyio.to_thread.run_sync(lambda: call(sim.say, svc, text[:1500]))
        out.setdefault("tool", None)
        return JSONResponse(out)

    async def advance(request) -> Response:
        data = await body(request)
        days = data.get("days") if data else None
        if not isinstance(days, int) or isinstance(days, bool):
            return JSONResponse({"ok": False}, status_code=400 if data is not None else 415)
        return JSONResponse({"ok": True, "skipped_days": sim.advance(days)})

    async def state(_r) -> Response:
        return JSONResponse(await anyio.to_thread.run_sync(lambda: call(sim.state, svc)))

    app.add_route("/sim", page)
    app.add_route("/sim/say", say, methods=["POST"])
    app.add_route("/sim/advance", advance, methods=["POST"])
    app.add_route("/sim/state", state)


def create_app(service: Tripwire | None = None, simulator: Simulator | None = None):
    sim = simulator or (Simulator() if os.environ.get("TRIPWIRE_SIM") == "1" else None)
    if service is None:
        svc = default_service()
        if sim is not None:  # simulated clock so the UI can "skip days"
            svc.today = sim.today
            svc._today = sim.today
    else:
        svc = service
    mcp = build_server(svc)
    hosts = [h.strip() for h in os.environ.get("TRIPWIRE_ALLOWED_HOSTS", "").split(",") if h.strip()]
    # Keep loopback allowed so local health checks and clients still work when a public hostname is configured.
    security = (TransportSecuritySettings(enable_dns_rebinding_protection=True,
                                          allowed_hosts=hosts + ["127.0.0.1:*", "localhost:*", "[::1]:*"]) if hosts else None)
    app = mcp.streamable_http_app(stateless_http=True, transport_security=security,
                                  host=os.environ.get("TRIPWIRE_HOST", "127.0.0.1"))

    async def healthz(_request) -> Response:
        return JSONResponse({"status": "ok"})

    async def readyz(_request) -> Response:
        """Readiness: can we actually reach the database? (/healthz only says the process is up.)"""
        try:
            await anyio.to_thread.run_sync(svc.s.ping)
        except Exception:  # noqa: BLE001
            log.exception("readiness check failed")
            return JSONResponse({"status": "unavailable"}, status_code=503)
        return JSONResponse({"status": "ready"})

    if sim is not None:
        _add_sim_routes(app, svc, sim)

    app.add_route("/healthz", healthz)
    app.add_route("/readyz", readyz)
    token = os.environ.get("TRIPWIRE_TOKEN")
    return BearerAuth(app, token) if token else app


def main() -> None:
    import uvicorn

    logging.basicConfig(level=os.environ.get("TRIPWIRE_LOG", "INFO"), format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    host = os.environ.get("TRIPWIRE_HOST", "127.0.0.1")
    port = int(os.environ.get("PORT", os.environ.get("TRIPWIRE_PORT", "8000")))
    if host not in ("127.0.0.1", "localhost", "::1"):
        if not os.environ.get("TRIPWIRE_ALLOWED_HOSTS"):
            log.warning("Listening on %s but TRIPWIRE_ALLOWED_HOSTS is unset: requests addressed to your public domain "
                        "will be rejected with HTTP 421. Set it to your hostname.", host)
        if not os.environ.get("TRIPWIRE_TOKEN"):
            log.warning("TRIPWIRE_TOKEN is unset: anyone who can reach this endpoint can read and change your decisions.")
    if os.environ.get("TRIPWIRE_AI", "auto").lower() != "offline" and not os.environ.get("BEDROCK_MODEL_ID"):
        log.warning("BEDROCK_MODEL_ID is unset: Bedrock is unavailable, so %s.",
                    "every request will fail" if os.environ.get("TRIPWIRE_AI", "auto").lower() == "bedrock" else "the offline fallback parser will be used")
    uvicorn.run(create_app(), host=host, port=port, timeout_keep_alive=5)


if __name__ == "__main__":
    main()

## 3. Configure
Runs fully offline by default. To use real AWS in section 7, add these in Colab's 🔑 **Secrets** panel (left sidebar) and
enable *Notebook access*: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION`, `BEDROCK_MODEL_ID`
(and `AWS_SESSION_TOKEN` if you use temporary credentials). **Never paste keys into a cell.**

In [ ]:
import os, sys, asyncio, threading, time
sys.path.insert(0, "tripwire_colab/src")

# Pull optional AWS settings from Colab Secrets (silently skipped if absent / not in Colab)
try:
    from google.colab import userdata
    for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_REGION", "BEDROCK_MODEL_ID"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("TRIPWIRE_AI", "offline")       # offline | auto | bedrock
os.environ.setdefault("TRIPWIRE_STORE", "sqlite")     # sqlite | dynamodb
os.environ["TRIPWIRE_DB"] = "/tmp/tripwire_demo.db"
os.environ["TRIPWIRE_HOUSEHOLD"] = "colab"
os.environ["TRIPWIRE_SIM"] = "1"                     # also serve the simulated Alexa+ web UI at /sim
PORT = 8000
URL = f"http://127.0.0.1:{PORT}/mcp"
print("AI engine:", os.environ["TRIPWIRE_AI"], "| store:", os.environ["TRIPWIRE_STORE"])

## 4. Start the MCP server
A real Starlette/uvicorn server on a background thread, speaking MCP over Streamable HTTP. Re-run this cell any time to restart it with a fresh demo database.

In [ ]:
import uvicorn, importlib
_state = {"server": None, "thread": None}

def stop_server():
    if _state["server"]:
        _state["server"].should_exit = True
        _state["thread"].join(timeout=10)
        _state["server"] = _state["thread"] = None

def start_server(reset=False):
    stop_server()
    if reset and os.path.exists(os.environ["TRIPWIRE_DB"]):
        os.remove(os.environ["TRIPWIRE_DB"])
    import tripwire.server as srv
    importlib.reload(srv)                      # pick up new env vars (store, AI mode, allowed hosts)
    server = uvicorn.Server(uvicorn.Config(srv.create_app(), host="127.0.0.1", port=PORT, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(100):
        if server.started:
            break
        time.sleep(0.1)
    else:
        raise RuntimeError(f"Server did not start (is port {PORT} already in use? Try Runtime > Restart session).")
    _state.update(server=server, thread=thread)
    print(f"✅ Tripwire MCP server running at {URL}")

start_server(reset=True)

## 5. 🎬 The demo conversation
Each line below is what a person says to Alexa+; Alexa+ turns it into an MCP tool call, and the server answers with a short
`spoken` sentence. Notice: the **new fact** about the price is matched to a decision made in an earlier conversation, and the
sofa is correctly left alone.

In [ ]:
from mcp import Client

STORY = [
    ("Alexa, remember: we're going with the 200 megabit internet plan because it's cheapest for our usage. "
     "We'd switch if the price goes above $50 or if outages hit more than twice a month. Check back in three months.",
     "record_decision", {"what_was_said": "We're going with the 200 megabit internet plan because it's cheapest for our usage. "
                         "We'd switch if the price goes above $50 or if outages hit more than twice a month. Check back in three months."}),
    ("Alexa, we're getting the grey sofa because it's comfy. We'd return it if the price drops below $300.",
     "record_decision", {"what_was_said": "We're going with the grey sofa because it's comfy. We'd return it if the price drops below $300."}),
    ("(two weeks later, NEW conversation) The provider emailed: our internet price goes to $48 next month.",
     "report_fact", {"statement": "The provider emailed that our internet price goes to $48 next month"}),
    ("(a week later) The provider just raised our internet price to $55.",
     "report_fact", {"statement": "The provider just raised our internet price to $55"}),
    ("Alexa, what should I look at?", "whats_on_watch", {}),
    ("Alexa, why did we pick that internet plan?", "why_did_we", {"topic": "internet plan"}),
    ("Alexa, keep it, but move the price limit to $60.", "resolve_decision",
     {"topic": "internet", "outcome": "keep", "note": "still best value", "new_limit": 60, "tripwire_number": 1}),
    ("Alexa, the sofa is on sale, the price dropped to $250.", "report_fact", {"statement": "The sofa price dropped to $250"}),
]

async def story():
    async with Client(URL) as c:                       # a real MCP client over HTTP
        print("negotiated MCP protocol revision:", c.protocol_version)
        for said, tool, args in STORY:
            r = await c.call_tool(tool, args)
            print(f"\n🗣  {said}\n   ↳ tool: {tool}\n🔊 {r.structured_content['spoken']}")

await story()

### Your turn
Edit the text and run. Phrase it your own way, e.g. *"We're going with the standing desk because it's on sale. We'd return it if
the delivery fee goes above $40."* then *"The delivery fee for the desk is $45."* (Offline mode uses a deliberately small
parser: it handles "X goes above/below N" style conditions. With Bedrock, section 7, free-form phrasing works much better.)

In [ ]:
my_decision = "We're going with the annual gym membership because it's cheaper per visit. We'd cancel if the fee goes above $60."
my_fact     = "The gym just told us the fee is going up to $65."

async def ask(tool, **args):
    async with Client(URL) as c:
        r = await c.call_tool(tool, args)
        print("🔊", r.structured_content["spoken"])
        return r.structured_content

_ = await ask("record_decision", what_was_said=my_decision)
_ = await ask("report_fact", statement=my_fact)
_ = await ask("whats_on_watch")

### 5b. The simulated Alexa+ screen *(optional)*
The same server also serves a **simulated Alexa+ experience** at `/sim`: type or speak, watch the decisions and gauges update, and press
**Skip 7 days** to simulate time passing. It is a simulation (simple routing rules stand in for Alexa+'s AI); all logic is the real Tripwire code.
_(I could only test this page locally, not inside Colab's iframe proxy. If the embedded view doesn't load, run the repo locally and open `http://127.0.0.1:8000/sim`.)_

In [ ]:
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, path="/sim", height=760)
except ImportError:
    print("Not running in Colab: open http://127.0.0.1:%d/sim in your browser instead." % PORT)

## 6. 🛡️ Safety proof: the model proposes, the server disposes
Here we **replace the AI with a liar**. It claims a $45 price *tripped* a "$50" tripwire, then claims a $70 price did *not*.
Because numeric thresholds are compared by server code, neither lie changes anything.

In [ ]:
from datetime import date
import tripwire.ai as ai
from tripwire.service import Tripwire
from tripwire.store import SqliteStore

svc = Tripwire(SqliteStore(":memory:"), household="safety-demo", today=lambda: date(2026, 10, 13))
svc.record_decision("We're going with the 200 megabit internet plan because it's cheapest. We'd switch if the price goes above $50.")

real = ai.assess_fact
def lying_model(fact, items):
    value = 45.0 if "45" in fact else 70.0
    verdict = "tripped" if value == 45.0 else "not_tripped"          # both verdicts are WRONG
    return ([{"tripwire_id": items[0]["id"], "observed_value": value, "verdict": verdict, "note": "trust me"}], "lying-model")

ai.assess_fact = lying_model
try:
    r1 = svc.report_fact("internet price is $45")
    print("Model said TRIPPED for $45 vs limit $50 ->", "server tripped:", bool(r1["tripped"]), "|", r1["spoken"])
    r2 = svc.report_fact("internet price is $70")
    print("Model said NOT TRIPPED for $70 vs limit $50 ->", "server tripped:", bool(r2["tripped"]), "|", r2["spoken"])
    assert not r1["tripped"] and r2["tripped"]
    print("\n✅ The server ignored both lies and applied the real comparison.")
finally:
    ai.assess_fact = real

## 7. *(Optional)* Real Amazon Bedrock + DynamoDB
Needs the Colab Secrets from section 3 and a **Bedrock model you have access to** in your region. This switches the server to:
Bedrock (Converse + forced tool use) for understanding, and **DynamoDB** for memory, then replays the story.
It creates a small on-demand table named `Tripwire` if it doesn't exist (cost: pennies).

In [ ]:
import boto3

need = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION", "BEDROCK_MODEL_ID")
if not all(os.environ.get(k) for k in need):
    print("⏭️  Skipping: add", ", ".join(k for k in need if not os.environ.get(k)), "to Colab Secrets to enable this section.")
else:
    ddb = boto3.client("dynamodb", region_name=os.environ["AWS_REGION"])
    try:
        ddb.describe_table(TableName="Tripwire")
    except ddb.exceptions.ResourceNotFoundException:
        ddb.create_table(
            TableName="Tripwire", BillingMode="PAY_PER_REQUEST",
            KeySchema=[{"AttributeName": "pk", "KeyType": "HASH"}, {"AttributeName": "sk", "KeyType": "RANGE"}],
            AttributeDefinitions=[{"AttributeName": "pk", "AttributeType": "S"}, {"AttributeName": "sk", "AttributeType": "S"}])
        ddb.get_waiter("table_exists").wait(TableName="Tripwire")
    # a throw-away household per run, deleted afterwards, so re-running never meets stale duplicates
    household = f"colab-{int(time.time())}"
    os.environ.update(TRIPWIRE_STORE="dynamodb", TRIPWIRE_TABLE="Tripwire", TRIPWIRE_AI="bedrock", TRIPWIRE_HOUSEHOLD=household)
    start_server()
    try:
        await story()                  # same conversation, now on real Bedrock + DynamoDB
        print("\n✅ If you saw sensible answers above, Bedrock + DynamoDB both work. (TRIPWIRE_AI=bedrock never falls back silently.)")
    finally:
        from tripwire.store import make_store
        print(f"🧹 cleaned up {make_store().purge(household)} item(s) from DynamoDB")

## 8. *(Optional)* Public HTTPS URL for Alexa+
Alexa+ needs a **remote HTTPS** MCP URL. This opens a free temporary [Cloudflare quick tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/)
to the server above.

⚠️ **For demos only.** The URL changes every run and dies when the Colab session ends, so for your final submission deploy
the repo somewhere permanent (see the README). Anyone with the URL can use the server; set `USE_TOKEN = True` to require a
bearer token, **only if your Alexa+ configuration can send one**.
_(I couldn't test this cell in my build sandbox because it can't reach Cloudflare, so report any problem you hit.)_

In [ ]:
import subprocess, re, secrets, urllib.request, queue, httpx

USE_TOKEN = False
if not os.path.exists("cloudflared"):
    try:
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    except Exception as e:
        raise RuntimeError(f"Could not download cloudflared ({e}). Check Colab's internet access and re-run.")
    os.chmod("cloudflared", 0o755)

proc = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                        stderr=subprocess.PIPE, text=True)
lines = queue.Queue()
threading.Thread(target=lambda: [lines.put(l) for l in proc.stderr], daemon=True).start()   # also keeps the pipe drained

host, deadline = None, time.time() + 60
while host is None and time.time() < deadline:
    try:
        m = re.search(r"https://([a-z0-9-]+\.trycloudflare\.com)", lines.get(timeout=1))
        host = m.group(1) if m else None
    except queue.Empty:
        if proc.poll() is not None:
            raise RuntimeError("cloudflared exited early; re-run this cell.")
if host is None:
    proc.terminate()
    raise RuntimeError("Tunnel didn't come up within 60s; re-run this cell.")

os.environ["TRIPWIRE_ALLOWED_HOSTS"] = host             # let the server accept the tunnel's Host header
if USE_TOKEN:
    os.environ["TRIPWIRE_TOKEN"] = secrets.token_urlsafe(24)
start_server()
time.sleep(3)
print("health check:", httpx.get(f"https://{host}/healthz", timeout=20).text)
print("\n🌐 MCP server URL for Alexa+:  https://%s/mcp" % host)
if USE_TOKEN:
    print("🔑 Bearer token:", os.environ["TRIPWIRE_TOKEN"])
print("\nNext: alexa-ai new mcp --name Tripwire --locale en-US --mcp-server-url https://%s/mcp" % host)

---
**Cleanup:** `stop_server()` stops the server. If you created the DynamoDB table in section 7 and no longer need it, delete it with
`boto3.client("dynamodb").delete_table(TableName="Tripwire")`.

**The full repo** (tests, CloudFormation template, Dockerfile, docs) is what you submit; this notebook is a convenient way to run and show it.